**Projet ISD2 final:** Analyse d'une base de données Formule 1! (2000-2024)

In [125]:
# importation des bibiliothéques nécéssaires au projet 
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt, ceil

In [110]:
# chargement des fichiers de source

Data_Dir = "DataSet_F1"                                 # pointe vers le fichier source du répositoire
Output = "DataSet_F1_Final"        
NA_VALUES = ["//N", ""] # nom du fichier complet

In [111]:
def load_csv(csv_file_name):
    file_path = os.path.join(Data_Dir, csv_file_name)
    return pd.read_csv(file_path, na_values = NA_VALUES)


In [112]:
def calculate_age(date_of_birth, current_year = 2026):
    year_of_birth = int(date_of_birth[:4])
    return current_year - year_of_birth

In [194]:
results           = load_csv("results.csv")
races             = load_csv("races.csv")
circuits          = load_csv("circuits.csv")
constructors      = load_csv("constructors.csv")
status            = load_csv("status.csv")
qualifying        = load_csv("qualifying.csv")
pit_stops         = load_csv("pit_stops.csv")
driver_standings  = load_csv("driver_standings.csv")
lap_times         = load_csv("lap_times.csv")
drivers           = load_csv("drivers.csv")

drivers["driver_name"] = drivers["forename"] + " " + drivers["surname"]
drivers["driver_age"] = drivers["dob"].apply(calculate_age)

pit_agg = pit_stops.groupby(["raceId", "driverId"]).agg(best_lap_ms = ("milliseconds", "min"), lap_std_ms = ("milliseconds", "std")).reset_index()
pits_aux  = pit_stops.groupby(["raceId", "driverId"]).count().reset_index()
pit_agg["pit_stop_count"] = pits_aux["stop"]



df = results.copy()
df = df.merge(races[["raceId", "year", "round", "circuitId", "date"]], on = "raceId", how = "left")
df = df.merge(circuits[["circuitId", "country", "name"]].rename(columns = {"name" : "circuit_name"}), on = "circuitId", how = "left")
df = df.merge(drivers[["driverId", "driver_age","driver_name" ]], on = "driverId", how = "left")
df = df.merge(constructors[["constructorId", "name"]].rename(columns = {"name" : "constructor_name"}), on = "constructorId", how = "left")
df = df.merge(status.rename(columns = {"status": "status_label"}), on = "statusId", how = "left")

df = df.merge(pit_agg, on = ["raceId", "driverId"], how = "left")
df["pit_stop_count"] = df["pit_stop_count"].fillna(0)
df = df.astype({"pit_stop_count" : "int32"})
# on renomme chacune des colonnes(optionel mais pratique)

df = df.rename(columns = {
    "grid":         "grid_position",
    "positionOrder":  "finish_position",
    "points":       "points_scored",
    "laps":         "laps_completed",
    "status_label": "status",
    "country":      "circuit_country",
    "rank":         "fastest_lap_rank",
    
})

# choix de nos features -> 12 en total 

features = [
    "year",                                     
    "round", 
    "grid_position", 
    "finish_position",
    "points_scored",
    "laps_completed",
    "status",
    "constructor_name",
    "circuit_country",
    "circuit_name",
    "driver_name",
    "driver_age",
    "pit_stop_count",
    "fastest_lap_rank",
]

df = df[features]
df = df[df["year"].between(2012, 2024)]        # reduit le nombre de lignes: passage de 1950-2024 à 2000-2024


df.to_csv(Output, index = False)

In [195]:
#visualistation des données

In [196]:
df.corr(numeric_only=True)["finish_position"]

year              -0.078926
round             -0.007109
grid_position      0.576891
finish_position    1.000000
points_scored     -0.836094
laps_completed    -0.486139
driver_age        -0.044530
pit_stop_count    -0.138483
Name: finish_position, dtype: float64

A premiere vue on dirait qu'il n'y a pas vraiment d'attribut en correlation
directe avec la position finale du pilote a part la position de depart "grid_position"

In [202]:
def calcule_dimensions_subplot(attributs):
    nombre_attributs = len(attributs)
    side = int(ceil(sqrt(nombre_attributs) ))
    return side, side

def calcule_subplot_index(index, width):
    y = index // width
    x = index % width
    return x , y
    
def affiche_graphiques(attributs, y_axis = "finish_position",chart_width = 7,chart_height = 7,):
    plot_width, plot_height = calcule_dimensions_subplot(attributs)
    fig , axes = plt.subplots(plot_width, plot_height, squeeze=False)
    fig.set_size_inches(plot_width * chart_width, plot_height * chart_height)
    
    for i, attribut in enumerate(attributs):
        x, y = calcule_subplot_index(i, plot_width)
        plot = axes[y,x]
        plot.set_title(f"{attribut} vs {y_axis}")
        plot.scatter(df[attribut], df[y_axis])
        plot.set_ylabel(f"{y_axis}")
        plot.set_xlabel(f"{attribut}")


In [ ]:

affiche_graphiques(df.columns, chart_width = 8, chart_height = 8)